# 0. SETUP

In [1]:
import os 
os.environ["PIP_QUIET"] = "1"

In [2]:
# Install required libraries and Run them only once
!pip install pandas openpyxl torch transformers scikit-learn matplotlib seaborn wandb
!pip install bertopic sentence-transformers wordcloud
!pip install nltk beautifulsoup4 contractions emoji pyspellchecker langdetect


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import pandas as pd
from nltk.collocations import BigramCollocationFinder
from nltk.collocations import BigramAssocMeasures
import nltk
import numpy as np
import matplotlib.pyplot as plt
import pyLDAvis
import pyLDAvis.gensim_models
from gensim import corpora
from gensim.models import LdaModel
import re
import argparse
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
# Plotly imports
import plotly.offline as py
py.init_notebook_mode(connected=True)
import plotly.graph_objs as go
import plotly.tools as tls
from matplotlib import pyplot as plt
%matplotlib inline

/Users/khnsakhnm/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


# 1. Loading Train and Test data 

In [4]:
import sys
!{sys.executable} -m pip install openpyxl

You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [5]:
import openpyxl

train_1 = pd.read_excel("/Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/data/raw_input/Coding - Gavin.xlsx")
train_2 = pd.read_excel("/Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/data/raw_input/Coding - Katie Complete 2.18.26.xlsx")

train = pd.concat([train_1, train_2], ignore_index=True)

cols = ['Functional', 'Relational', 'Metaphysical', 'Technical']
train[cols] = train[cols].apply(pd.to_numeric, errors='coerce').astype('Int64')
train.columns = train.columns.str.lower()
train.drop(columns=['relational', 'metaphysical', 'technical'], inplace=True)

train.head()

,text,functional
0,this whole thread is rough to read. i think ev...,0
1,I'm developing a solve for exactly this - a jo...,1
2,I don't know if this has been reported here ye...,99
3,https://preview.redd.it/lj49iq34it9f1.jpeg?wid...,99
4,https://preview.redd.it/4mfmbgmd3kcg1.jpeg?wid...,99


In [6]:
train.to_csv("/Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/data/processed_input/train.csv", index=False)

In [7]:
test = pd.read_csv("/Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/data/raw_input/cleaned_combined_posts_comments.csv")
test.head()
test.columns = test.columns.str.lower()
test.to_csv("/Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/data/processed_input/test.csv", index=False)

In [8]:
test.head()

,id,parent_id,type,text,score,created_utc,index
0,1jvg6gw,NaN,post,,1,1744231180,1
1,1jx1pxy,NaN,post,this space isnt about replacing therapists its...,1,1744408477,2
2,1jxkyee,NaN,post,one of the weirdest and most useful things abo...,4,1744474650,6
3,1jzpmtb,NaN,post,hey im mike i started this subreddit because i...,8,1744717048,7
4,1k10bvs,NaN,post,ive been in and out of therapy for most of my ...,15,1744851085,8


# 2. Data Pre-Processing

In [24]:
base_path = os.getcwd()
train_path = "/Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/data/processed_input/train.csv"
test_path = "/Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/data/processed_input/test.csv"

In [10]:
'''
Reads and cleans training and test CSV data for the
AI mental health Reddit study.

INPUTS:
1. data/train.csv
2. data/test.csv

LABEL COLUMNS (train only):
{Functional, Relational, Metaphysical, Technical}
Each coded as: 1=positive  0=neutral  -1=negative  99=not_mentioned
'''

%run run_preprocessing.py --train_path {train_path} --test_path {test_path}

# Comment out the above line after running once to avoid re-processing every time you run this notebook

Preprocessor: TextPreprocessor(use_stemming=False, use_lemma=True, remove_stopwords=True, expand_contractions=True, handle_emojis=True, handle_negation=True, handle_chatwords=True, spell_check=False, language='english')

Loading training data from /Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/data/processed_input/train.csv...
  Shape: (801, 2)
  Removed/deleted : 1
  Image-only      : 5
  Running TextPreprocessor on training data...
  Flagged 133 posts as too short.
  Sample:
0    whole thread rough read think everyone look be...
1    developing solve exactly journalling tool prov...
2    not_know reported yet also little annoyed new ...
 Processed training data: 

Loading test data from /Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/data/processed_input/test.csv...
  Shape: (19240, 7)
  Removed/deleted : 2
  Image-only      : 0
  Running TextPreprocessor on test data in batches of 1000...
  Processed 1000 / 19240 rows...
  Processed 2000 / 19240 rows...


In [11]:
# Quick Sanity Check 
train_df = pd.read_csv(train_path)

print(f"Train rows: {train_df.shape[0]} \n")
print(f"Train columns: {train_df.shape[1]} \n")
print(train_df.head())

Train rows: 801 

Train columns: 6 

                                                text  functional  is_removed  \
0  this whole thread is rough to read. i think ev...         0.0       False   
1  I'm developing a solve for exactly this - a jo...         1.0       False   
2  I don't know if this has been reported here ye...        99.0       False   
3  https://preview.redd.it/lj49iq34it9f1.jpeg?wid...        99.0       False   
4  https://preview.redd.it/4mfmbgmd3kcg1.jpeg?wid...        99.0       False   

   is_image                                       cleaned_text  too_short  
0     False  whole thread rough read think everyone look be...      False  
1     False  developing solve exactly journalling tool prov...      False  
2     False  not_know reported yet also little annoyed new ...      False  
3      True                                                NaN       True  
4      True                                                NaN       True  


In [12]:
# Quick Sanity Check
test_df = pd.read_csv(test_path)

print(f"Test rows: {test_df.shape[0]} \n")
print(f"Test columns: {test_df.shape[1]} \n")   
print(test_df.head())

Test rows: 19240 

Test columns: 11 

        id parent_id  type                                               text  \
0  1jvg6gw       NaN  post                                                      
1  1jx1pxy       NaN  post  this space isnt about replacing therapists its...   
2  1jxkyee       NaN  post  one of the weirdest and most useful things abo...   
3  1jzpmtb       NaN  post  hey im mike i started this subreddit because i...   
4  1k10bvs       NaN  post  ive been in and out of therapy for most of my ...   

   score  created_utc  index  is_removed  is_image  \
0      1   1744231180      1        True     False   
1      1   1744408477      2       False     False   
2      4   1744474650      6       False     False   
3      8   1744717048      7       False     False   
4     15   1744851085      8       False     False   

                                        cleaned_text  too_short  
0                                                NaN       True  
1  space not_about

# 3. Exploratory Data Analysis

In [13]:
%run EDA.py \
    --train {train_path} \
    --test {test_path}


Loading /Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/data/processed_input/train.csv...
  Shape: (801, 6)

Loading /Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/data/processed_input/test.csv...
  Shape: (19240, 11)

[1/10] Label distribution (train)...
  ✓ /Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/Plots/functional_class_distribution.png

  Label breakdown:
    positive          :  192  (24.0%)
    neutral           :   56  (7.0%)
    negative          :   54  (6.7%)
    not_mentioned     :  497  (62.0%)

[2/10] Text length — train vs test...
  ✓ /Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/Plots/text_length_train_vs_test.png

  Token length stats:
    Train  mean=38.8  median=19.0  max=584
    Test  mean=39.1  median=19.0  max=2865

[3/10] Token length by label (train)...
  ✓ /Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/Plots/token_length_by_label.png

  Median token count by label:
    positive     

<Figure size 640x480 with 0 Axes>

# 4. Classification (using HuggingFace's mental/mental-roberta-base)

In [14]:
import torch 
print(f"Is CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
else:
    print("GPU not available")

Is CUDA available: False
GPU not available


In [ ]:
output_dir = "/Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/data/output_dir"
data_dir   = "/Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/data/processed_input"
num_epochs = 10
hf_token   = "YOUR_HF_TOKEN"

In [16]:
cmd = (
    f"classification.py"
    f" --train      {train_path}"
    f" --test       {test_path}"
    f" --output_dir {output_dir}"
    f" --data_dir   {data_dir}"
    f" --epochs     {num_epochs}"
    f" --hf_token   {hf_token}"
)

%run $cmd

Device: cpu

 Loading Training Data 
 Shape of Training Data: (801, 6)

 Loading Test Data 
 Shape of Test Data: (19240, 11)
Shape of Training Data after Filtering: (667, 6)
       negative ( -1): 54
        neutral (  0): 56
       positive (  1): 190
  not_mentioned ( 99): 367
 SUMMARY: 
Training Dataset: 533Testing Dataset: 134

 Loading tokenizer (mental/mental-roberta-base)
Loading Model: (mental/mental-roberta-base)


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at mental/mental-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Class weights: {'negative': '3.10', 'neutral': '2.96', 'positive': '0.88', 'not_mentioned': '0.45'}
 Training for 10 epochs
 Epoch 1/10 train_loss = 1.3921 | train_accuracy = 0.2364 val_loss = 1.3565 | val_accuracy = 0.4030
 Best model so far saved to /Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/data/output_dir/best_model best validation loss = 1.3565
 Epoch 2/10 train_loss = 1.3459 | train_accuracy = 0.4015 val_loss = 1.3312 | val_accuracy = 0.5522
 Best model so far saved to /Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/data/output_dir/best_model best validation loss = 1.3312
 Epoch 3/10 train_loss = 1.2226 | train_accuracy = 0.6041 val_loss = 1.3102 | val_accuracy = 0.6045
 Best model so far saved to /Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/data/output_dir/best_model best validation loss = 1.3102
 Epoch 4/10 train_loss = 1.0811 | train_accuracy = 0.5741 val_loss = 1.4130 | val_accuracy = 0.5149
 Epoch 5/10 train_loss = 0.9048 | t

<Figure size 640x480 with 0 Axes>

In [17]:
test_df = pd.read_csv("/Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/data/processed_input/test_withPred.csv")
lowconf_df = pd.read_csv("/Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/data/processed_input/low_conf_review.csv")

In [18]:
len(test_df)

19240

In [19]:
len(lowconf_df)

8811

# 5. Low Confidence Review using CatLLM 

In [7]:
# CatLLM is compatible on Python versions over 3.10+
import sys
print(sys.version)

3.12.2 | packaged by conda-forge | (main, Feb 16 2024, 20:54:21) [Clang 16.0.6 ]


In [8]:
!pip install groq


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [9]:
!pip install cat-llm 


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
pred_data_path = "/Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/data/processed_input/test_withPred.csv"
lowconf_data_path = "/Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/data/processed_input/low_conf_review.csv"
data_dir   = "/Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/data/processed_input"
llm_model = "llama-3.1-8b-instant"
groq_key = "YOUR_GROQ_KEY"

In [14]:
cmd = (
    f"lowconf_data_review.py"
    f" --predictions {pred_data_path}"
    f" --low_conf    {lowconf_data_path}"
    f" --output_dir  {data_dir}"
    f" --llm_model   llama-3.1-8b-instant"
    f" --groq_key  {groq_key}"
)
%run $cmd


 CatLLM Review 


 Loading Full Predictions -> /Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/data/processed_input/test_withPred.csv
 Shape : (19240, 15)
  Loading low-conf subset -> /Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/data/processed_input/low_conf_review.csv
 Shape: (8811, 15)  (45.8% of test set)

  Running CatLLM (llama-3.1-8b-instant) on 8,811 low-confidence rows...
  Resuming from row 8811 (8811 rows already done)

  CatLLM predicted distribution:
catllm_label
not_mentioned    8760
negative           21
neutral            18
positive           12

 Agreement between mental/mental-roberta-base and CatLLM:
 Total Compared: 8,811
 Agree : 7,388 (83.8%)
 Disgree : 1,423  (16.2%)

 Disagreement Breakdown: 
roberta_label  catllm_label 
positive       not_mentioned    762
negative       not_mentioned    471
neutral        not_mentioned    149
not_mentioned  neutral           11
               negative          10
               positive          

In [15]:
test_df = pd.read_csv("/Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/data/processed_input/test_finalPreds.csv")

In [20]:
print(f"\n SUMMARY: \n"
      f"\n 1. Length of train dataset: {len(pd.read_csv("/Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/data/processed_input/train.csv"))} \n"
      f"\n 2. Length of inital test dataset: {len(pd.read_csv("/Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/data/processed_input/test.csv"))} \n"
      f"\n 3. Length of test dataset before CatLLM classification (with undertainity): {len(pd.read_csv("/Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/data/processed_input/test_withPred.csv"))} \n"
      f"\n 4. Length of uncertain rows in test dataset: {len(pd.read_csv("/Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/data/processed_input/low_conf_review.csv"))} \n"
      f"\n 5. Length of rows classified by CatLLM: {len(pd.read_csv("/Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/data/processed_input/catllm_review.csv"))} \n"
      f"\n 6. Length of final test dataset after verified classification: {len(test_df)} \n"
      )


 SUMMARY: 

 1. Length of train dataset: 801 

 2. Length of inital test dataset: 19240 

 3. Length of test dataset before CatLLM classification (with undertainity): 19240 

 4. Length of uncertain rows in test dataset: 8811 

 5. Length of rows classified by CatLLM: 8811 

 6. Length of final test dataset after verified classification: 19240 



# 6. Annotation Agreement (Human Annotations (vs) CatLLM classification)

*As ground truth labels are unavailable for the test set, model reliability was assessed through two complementary approaches:*
- *(1) inter-annotator agreement between human coders and CatLLM on the labeled training set (Cohen's κ = X), and*
- *(2) validation set performance of MentalRoBERTa (macro F1 = X). Low-confidence predictions (< 0.40, n = 8,811, 45.8%) were additionally reviewed by CatLLM to improve label quality.*

In [21]:
!pip install groq krippendorff scikit-learn


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [25]:
train_path = train_path
output_path = data_dir
groq_key = groq_key

In [26]:
cmd = (
    f"annotation_agreement.py"
    f" --train     {train_path}"
    f" --output    {output_path}"
    f" --groq_key  {groq_key}"
)
%run $cmd



  Annotation Agreement: Human vs CatLLM


  Loading training data from /Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/data/processed_input/train.csv...
  Total training samples: 801 (801 rows, 6 columns)

  Human label distribution:
human_label
not_mentioned    497
positive         192
neutral           56
negative          54
  Testing Groq API key...
  ✓ Key valid. Response: ok
  Processed 50/799 rows...
  Processed 100/799 rows...
  Processed 150/799 rows...
  Processed 200/799 rows...
  Processed 250/799 rows...
  Processed 300/799 rows...
  Processed 350/799 rows...
  Processed 400/799 rows...
  Processed 450/799 rows...
  Processed 500/799 rows...
  Processed 550/799 rows...
  Processed 600/799 rows...
  Processed 650/799 rows...
  Processed 700/799 rows...
  Processed 750/799 rows...

  CatLLM label distribution:
catllm_label
not_mentioned    364
positive         180
negative         130
neutral          125

  Labelled file saved → /Users/khnsakhnm/Documents/Me

# 7. Thematic Analysis 

In [27]:
!pip install bertopic sentence-transformers umap-learn hdbscan


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [28]:
input_path = "/Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/data/processed_input/test_finalPreds.csv"
plot_path = "/Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/Plots/thematic_analysis"
output_path = "/Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/Plots/thematic_analysis/output"

In [29]:
cmd = (
    f"thematic_analysis.py"
    f" --input      {input_path}"
    f" --plots      {plot_path}"
    f" --output     {output_path}"
)

%run $cmd


  Thematic Analysis — AI Mental Health Reddit

[1/6] Loading data from /Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/data/processed_input/test_finalPreds.csv...
  Using 'final_label' column (post CatLLM review)
  Loaded 19,186 rows with valid text.

[2/6] Plotting label distribution...
  ✓ /Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/Plots/thematic_analysis/label_distribution.png

[3/6] Plotting word frequency per label...
  ✓ /Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/Plots/thematic_analysis/word_frequency_per_label.png

[4/6] Plotting sentiment share over time...
  ✓ /Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/Plots/thematic_analysis/sentiment_over_time.png

[5/6] Running BERTopic per sentiment label...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


  [negative]  53 posts
    Found 0 topics  |  Outliers: 53

  [neutral]  265 posts
    Found 4 topics  |  Outliers: 133
  ✓ /Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/Plots/thematic_analysis/topics_neutral.png
  ✓ /Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/Plots/thematic_analysis/topic_words_neutral.png

  [positive]  5,505 posts
    Found 29 topics  |  Outliers: 3,031
  ✓ /Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/Plots/thematic_analysis/topics_positive.png
  ✓ /Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/Plots/thematic_analysis/topic_words_positive.png

  [not_mentioned]  13,363 posts
    Found 153 topics  |  Outliers: 6,101
  ✓ /Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/Plots/thematic_analysis/topics_not_mentioned.png
  ✓ /Users/khnsakhnm/Documents/MentalHealth_Research/AI4MH-Reddit/Plots/thematic_analysis/topic_words_not_mentioned.png

  Topic summary saved → /Users/khnsakhnm/Documents/Men

In [30]:
topics_df = pd.read_csv(os.path.join(output_path, "all_topics.csv"))
print(f"Total topics found: {len(topics_df)}")
topics_df.groupby("label")[["Topic", "Count", "Name"]].head(5)

Total topics found: 186


,Topic,Count,Name
0,0,35,0_truth_feel_even_fear
1,1,34,1_ai_therapy_therapist_people
2,2,32,2_therapist_therapy_also_people
3,3,31,3_people_thing_like_think
4,0,1064,0_ai_therapy_therapist_people
5,1,232,1_therapist_therapy_people_client
6,2,227,2_chatgpt_gpt_therapist_therapy
7,3,176,3_therapist_trauma_cptsd_therapy
8,4,83,4_life_year_therapy_work
33,0,608,0_therapist_therapy_human_one


1. https://www.geeksforgeeks.org/machine-learning/python-lemmatization-approaches-with-examples/
2. https://www.geeksforgeeks.org/machine-learning/introduction-to-stemming/
3. https://www.geeksforgeeks.org/dsa/write-regular-expressions/
4. https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0318464
5. https://huggingface.co/spaces/CatLLM/survey-classifier
